In [ ]:
# Setup
from pathlib import Path
import sys, json
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data import load_data
from model import split_data, build_tfidf_pipeline, evaluate_classifier, load_pipeline
from utils import save_fig, plot_confusion

ARTEFACTS_DIR = PROJECT_ROOT / 'artefacts'
with open(ARTEFACTS_DIR / 'nlp_results.json') as f:
    results = json.load(f)
print("Models evaluated:", list(results.keys()))

In [ ]:
# 1. Comparison table
rows = []
for model_name in ['tfidf_baseline', 'distilbert']:
    if model_name not in results or not results[model_name]:
        continue
    m = results[model_name]
    rows.append({
        'model': model_name,
        'macro_F1': f"{m['f1_macro']:.3f}",
        'weighted_F1': f"{m['f1_weighted']:.3f}",
        'precision_macro': f"{m['precision_macro']:.3f}",
        'recall_macro': f"{m['recall_macro']:.3f}",
    })
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# 2. Confusion matrix for the TF-IDF baseline
cm = results['tfidf_baseline']['confusion_matrix']
plot_confusion(cm)
save_fig('04_confusion_matrix.png')
plt.show()

In [ ]:
# 3. Top complaint themes (BERTopic)
topics = results.get('topics_negative', {})
if isinstance(topics, dict) and 'error' not in topics:
    print("Top complaint themes in negative reviews:")
    for tid, info in topics.items():
        print(f"  Topic {tid} ({info['count']} reviews): {', '.join(info['top_words'][:5])}")
else:
    print(f"BERTopic not available: {topics}")

In [ ]:
# 4. Business-readable summary
best_model = 'distilbert' if results.get('distilbert') else 'tfidf_baseline'
best_metrics = results[best_model]

summary_lines = [
    "=== BUSINESS SUMMARY ===",
    "Business question: monitor customer sentiment at scale to spot emerging issues",
    "before they show up in returns or complaints.",
    "",
    f"Best model: {best_model}",
    f"Macro F1: {best_metrics['f1_macro']:.3f}",
    f"Weighted F1: {best_metrics['f1_weighted']:.3f}",
    "",
    "Top complaint themes (BERTopic on negative reviews):",
]
if isinstance(topics, dict) and 'error' not in topics:
    for tid, info in list(topics.items())[:5]:
        summary_lines.append(f"  Topic {tid} ({info['count']} reviews): {', '.join(info['top_words'][:5])}")
else:
    summary_lines.append("  BERTopic not available on this run")

summary_lines += [
    "",
    "Recommended action:",
    "- Deploy the TF-IDF baseline as the real-time classifier (sub-ms inference).",
    "- Run DistilBERT weekly on the full review batch for higher-accuracy tracking.",
    "- Send the BERTopic themes to the product team weekly as an early-warning dashboard.",
]
print("\n".join(summary_lines))